In [ ]:
# Import packages and plotting setup
import pandas as pd
import numpy as np
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
from matplotlib.colors import to_rgb

from Functions import setup_matplotlib_parameter

wong_cycle = setup_matplotlib_parameter()

In [ ]:
# Load data
data_dir = Path("..") / "Data files" / "Inclusion lists per stage"

df_container = {}
for f in data_dir.glob("*.xlsx"):
    if "S1-DO" not in f.name:
        df_container[f.stem] = pd.read_excel(f, sheet_name="NMI data", index_col=0)

In [ ]:
# Conversion to oxides
# Molar masses (g/mol)
M_Al  = 26.98
M_Ca  = 40.08
M_Mg  = 24.31
M_S   = 32.06
M_O   = 15.99
M_Mn  = 54.94

M_Al2O3 = 2 * M_Al + 3 * M_O
M_CaO   = M_Ca + M_O
M_MgO   = M_Mg + M_O
M_CaS   = M_Ca + M_S
M_MnS   = M_Mn + M_S

# Conversion factors (mass of oxide per mass of element)
f_Al2O3 = M_Al2O3 / (2 * M_Al)  # divide by 2 because 2 Al per formula unit
f_MgO   = M_MgO   / M_Mg

def molar_to_weight(n_CaO, n_Al2O3):
    w_CaO   = n_CaO   * M_CaO
    w_Al2O3 = n_Al2O3 * M_Al2O3
    total   = w_CaO + w_Al2O3
    return w_CaO/total*100, w_Al2O3/total*100

def molar_to_weight_mgo(n_MgO, n_Al2O3):
    w_MgO   = n_MgO   * M_MgO
    w_Al2O3 = n_Al2O3 * M_Al2O3
    total   = w_MgO + w_Al2O3
    return w_MgO/total*100, w_Al2O3/total*100

# Reference lines for stages: stage_index -> (numerator, denominator, ref_atomic_ratio | None)
STAGE_RATIOS = {
    "S2-DS": ("Al2O3", "MgO", molar_to_weight_mgo(1, 1)[1] / molar_to_weight_mgo(1, 1)[0]),
    "S3-VT": ("Al2O3", "MgO", molar_to_weight_mgo(1, 1)[1] / molar_to_weight_mgo(1, 1)[0]),
    "S4-CT": ("Al2O3", "CaO", molar_to_weight(12, 7)[1] / molar_to_weight(12, 7)[0]),
    "S5-CT": ("Al2O3", "CaO", molar_to_weight(12, 7)[1] / molar_to_weight(12, 7)[0]),
    "S6-CT": ("Al2O3", "CaO", molar_to_weight(12, 7)[1] / molar_to_weight(12, 7)[0]),
    "S7-CT": ("Al2O3", "CaO", molar_to_weight(12, 7)[1] / molar_to_weight(12, 7)[0]),
    "S8-TU": ("Al2O3", "CaO", molar_to_weight(12, 7)[1] / molar_to_weight(12, 7)[0]),
}

HEATS_ORDER  = ["1", "2", "3", "4"]          # column order within each row
STAGE_LABELS = ["S2-DS", "S3-VT", "S4-CT", "S5-CT", "S6-CT", "S7-CT", "S8-TU"]
MASSES = {"Al":26.98,"Ca":40.08,"Mg":24.31,"Mn":54.94,"Si":28.09,"Ti":47.87, "S":32.06}
N_HEATS = len(HEATS_ORDER)

# storage[stage_idx][heat] = array of ln(NUM/DEN)
storage = {
    si: {h: np.array([]) for h in HEATS_ORDER}
    for si in STAGE_LABELS
}

In [ ]:
CLIP_Q = 0.99   # keep central 99% of each violin; None to disable

def clip_tails(r, q=CLIP_Q):
    """
    Clip the tails of the distribution.
    """
    if q is None or r.size == 0:
        return r
    lo, hi = np.quantile(r, [(1 - q) / 2, 1 - (1 - q) / 2])
    return r[(r >= lo) & (r <= hi)]

# Calculate log10 ratios
for key, df in df_container.items():
    stage_idx = key[:5]
    heat_idx = key.split("-")[-1]
    num, den, _ = STAGE_RATIOS[stage_idx]  # stage-specific elements

    # Generate mask with num, and den filter larger than 0.1 wt%
    mask = (df[num] > 0.001) & (df[den] > 0.001)
    cleaned_df = df.loc[mask, [num, den]].copy()
    r = np.log10(cleaned_df[num] / cleaned_df[den]).to_numpy()
    r = r[np.isfinite(r)] # drop zeros/inf first
    storage[stage_idx][heat_idx] = clip_tails(r)   # then trim the tails

In [ ]:
def lighten(color, amount=0.6):
    r, g, b = to_rgb(color)
    return (
        1 - amount*(1-r),
        1 - amount*(1-g),
        1 - amount*(1-b)
    )

sample_areas = {
    "S1-DO-1": 20.51, "S1-DO-2": 20.51, "S1-DO-3": 15.44, "S1-DO-4": 20.1,
    "S2-DS-1": 20.51, "S2-DS-2": 19.74, "S2-DS-3": 20.39, "S2-DS-4": 18.53,
    "S3-VT-1": 20.51, "S3-VT-2": 20.51, "S3-VT-3": 20.51, "S3-VT-4": 20.51,
    "S4-CT-1": 12.19, "S4-CT-2": 17.68,
    "S5-CT-1": 14.92, "S5-CT-2": 20.77,
    "S6-CT-1": 17.15, "S6-CT-2": 17.15,
    "S7-CT-1": 13.01, "S7-CT-2": 12.46, "S7-CT-3": 19.78, "S7-CT-4": 10.05,
    "S8-TU-1": 30.36, "S8-TU-2": 20.51, "S8-TU-3": 20.51, "S8-TU-4": 20.51,
}

In [ ]:
import string

fig, axes = plt.subplots(4, 2,
                         figsize=(210/25.4*0.8, 297/25.4*0.7),
                         sharex=True,
                         sharey="row")

ROW_LIMITS = (
    (-2, 3),
    (-2, 3),
    (-2, 3),
    (-1.5, 2.25),
)

ROW_TICKS = (
    (np.arange(-2, 3, 1)),
    (np.arange(-2, 3, 1)),
    (np.arange(-2, 3, 1)),
    (np.arange(-1.5, 1.75, 0.5)),

)

YLABELS = [
    r"$\log_{10}\left(\frac{{\mathrm{{{Al_2O_3}}}}}{{\mathrm{{{MgO}}}}}\right)$",
    r"$\log_{10}\left(\frac{{\mathrm{{{Al_2O_3}}}}}{{\mathrm{{{CaO}}}}}\right)$",
    r"$\log_{10}\left(\frac{{\mathrm{{{Al_2O_3}}}}}{{\mathrm{{{CaO}}}}}\right)$",
    r"$\log_{10}\left(\frac{{\mathrm{{{Al_2O_3}}}}}{{\mathrm{{{CaO}}}}}\right)$",
]

axs = axes.flatten()

for i, stage in enumerate(STAGE_LABELS):
    num, den, ref_ratio = STAGE_RATIOS[stage]

    if "S4-CT" in stage or "S5-CT" in stage or "S6-CT" in stage:
        pos = [0, 1]
    else:
        pos = [0, 1, 2, 3]

    vals = [storage[stage][HEATS_ORDER[j]] for j in pos]

    if vals:
        parts = axs[i].violinplot(vals, positions=pos, widths=0.75, showextrema=False)
        for pc, j in zip(parts["bodies"], pos):
            pc.set_alpha(None)
            pc.set_facecolor(lighten(wong_cycle[j], 0.45))
            pc.set_edgecolor(wong_cycle[j])
            pc.set_linewidth(1.2)
            pc.set_zorder(5)

            median = np.median(storage[stage][HEATS_ORDER[j]])

            axs[i].scatter(
                    j,
                    median,
                    s=12,
                    color="black",
                    zorder=5,
                )

    # x in data coords (over each heat), y in axes coords (same height every row)
    trans = mtransforms.blended_transform_factory(axs[i].transData, axs[i].transAxes)
    for j in pos:
        n = storage[stage][HEATS_ORDER[j]].size
        axs[i].text(j, 0.92, f"n={n}", transform=trans, ha="center", va="top", fontsize=8, color="0.35")
        measured_area = sample_areas[stage + "-" + HEATS_ORDER[j]]
        axs[i].text(j, 1.02, rf"$\mathrm{{\rho}}$={n / measured_area:.1f}", transform=trans, ha="center", va="top", fontsize=8, color="0.35")

    axs[i].grid(False)
    axs[i].set_axisbelow(False)
    axs[i].grid(True, axis="y", which="major", color="0.88", lw=0.7, zorder=0)

    axs[i].text(
        0.0,
        1.15,
        f"({string.ascii_lowercase[i]})",
        transform=axs[i].transAxes,
        ha="left",
        va="top",
        fontweight="bold",
    )

    if i == 1:
        axs[i].text(1.01, 0.30, r"$\mathrm{MgO}\cdot\mathrm{Al_2O_3}$", fontsize=9, color="0.35", rotation=90,
                transform=axs[i].transAxes)

    elif i == 3:
        axs[i].text(1.01, 0.32, r"$\mathrm{C_{12}A_{7}}$", transform=axs[i].transAxes, fontsize=9, color="0.35", rotation=90)

    elif i == 5:
        axs[i].text(1.01, 0.32, r"$\mathrm{C_{12}A_{7}}$", transform=axs[i].transAxes, fontsize=9, color="0.35", rotation=90)

    elif i == 6:
        axs[i].text(1.01, 0.32, r"$\mathrm{C_{12}A_{7}}$", transform=axs[i].transAxes, fontsize=9, color="0.35", rotation=90)

    axs[i].axhline(np.log10(ref_ratio), color="0.45", ls="--", lw=0.8, zorder=10)

# Remove top and right spines
for ax in axes.flatten():
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for row in range(4):
    axes[row, 0].set_ylabel(YLABELS[row], labelpad=15, va="center", fontsize=12)
    axes[row, 0].set_ylim(ROW_LIMITS[row])
    axes[row, 0].set_yticks(ROW_TICKS[row])

axes[3, 0].set_xticks(range(N_HEATS))
axes[3, 0].set_xticklabels([name.replace("H", "") for name in HEATS_ORDER])
axes[3, 0].set_xlabel("Heat", fontweight="normal", fontsize=12)

axes[2, 1].tick_params(axis="x", labelbottom=True)
axes[2, 1].set_xticks(range(N_HEATS))
axes[2, 1].set_xticklabels([name.replace("H", "") for name in HEATS_ORDER])
axes[2, 1].set_xlabel("Heat", fontweight="normal", fontsize=12)


fig.tight_layout()

fig.subplots_adjust(
    hspace=0.33,
    wspace=0.05,
)

from matplotlib.lines import Line2D

for row in range(3):
    ax_left  = axes[row, 0]
    ax_right = axes[row, 1]

    # Use the reference value of the left subplot
    if row == 0:
        num, den, ref_ratio = STAGE_RATIOS["S2-DS"]
    else:
        num, den, ref_ratio = STAGE_RATIOS["S4-CT"]

    # convert data y -> figure coordinates
    y_disp = ax_left.transData.transform((0, np.log10(ref_ratio)))[1]
    y_fig = fig.transFigure.inverted().transform((0, y_disp))[1]

    x0 = ax_left.get_position().x0
    x1 = ax_right.get_position().x1

    fig.add_artist(
        Line2D(
            [x0, x1],
            [y_fig, y_fig],
            transform=fig.transFigure,
            color="0.45",
            ls="--",
            lw=0.8,
            zorder=100,
        )
    )

axes[3, 0].text(
    1.28,
    0.5,
    r"$\mathrm{\rho}$...NMI density in $\mathrm{mm^{-2}}$",
    transform=axes[3, 0].transAxes, fontsize=9, color="0.35")

axes[3, 0].text(
    1.28,
    0.38,
    r"n...Number of NMIs",
    transform=axes[3, 0].transAxes, fontsize=9, color="0.35")

axes[3, 1].set_visible(False)

plt.show()